In [1]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
import lightgbm as lgb
import joblib
import time

In [2]:
df = pd.read_csv("../../datasets/training_dataset/train.csv")

In [3]:
X = df.drop(columns=['label'])
y = df['label']

In [4]:
print("\nTraining Decision Tree...")
start = time.time()
dt = DecisionTreeClassifier(max_depth=10, random_state=42)
dt.fit(X, y)
y_pred_dt = dt.predict(X)
end = time.time()
print(f"Decision Tree Training Time: {end - start:.2f} sec")


Training Decision Tree...
Decision Tree Training Time: 56.49 sec


In [5]:
print("\nDecision Tree Report:")
print("Accuracy:", accuracy_score(y, y_pred_dt))
print(classification_report(y, y_pred_dt))
joblib.dump(dt, "./models/decision_tree_model_with_no_feature_filtration.pkl")


Decision Tree Report:
Accuracy: 0.9026110416666666
              precision    recall  f1-score   support

           0       0.90      0.92      0.91   2696949
           1       0.90      0.88      0.89   2103051

    accuracy                           0.90   4800000
   macro avg       0.90      0.90      0.90   4800000
weighted avg       0.90      0.90      0.90   4800000



['./models/decision_tree_model_with_no_feature_filtration.pkl']

In [ ]:
print("\nTraining Random Forest...")
start = time.time()
rf = RandomForestClassifier(
    n_estimators=100,   # number of trees (default 100)
    max_depth=None,     # let trees expand fully
    n_jobs=-1,          
    random_state=42
)
rf.fit(X, y)
y_pred_rf = rf.predict(X)
end = time.time()
print(f"Random Forest Training Time: {end - start:.2f} sec")


Training Random Forest...
Random Forest Training Time: 336.97 sec


In [ ]:
print("\nRandom Forest Report:")
print("Accuracy:", accuracy_score(y, y_pred_rf))
print(classification_report(y, y_pred_rf))
# joblib.dump(rf, "./models/random_forest_model_with_no_feature_filtration.pkl")


Random Forest Report:
Accuracy: 0.9955745833333334
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   2696949
           1       1.00      0.99      0.99   2103051

    accuracy                           1.00   4800000
   macro avg       1.00      1.00      1.00   4800000
weighted avg       1.00      1.00      1.00   4800000



['./models/random_forest_model_with_no_feature_filtration.pkl']

Random forest is rejected though it gives good accuracy but is too much memory intensive.

In [6]:
train_data = lgb.Dataset(X, label=y)

params = {
    "objective": "binary",  
    "boosting": "gbdt",
    "metric": "binary_error", 
    "num_leaves": 64,
    "learning_rate": 0.1,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1
}

print("\nTraining LightGBM on full train.csv ...")
start = time.time()

# Train model (no validation split)
model = lgb.train(
    params,
    train_data,
    num_boost_round=500
)

end = time.time()
print(f"✅ LightGBM Training Time: {end - start:.2f} sec")

# Predictions on training data
y_pred = model.predict(X)
y_pred_binary = (y_pred > 0.5).astype(int)


Training LightGBM on full train.csv ...
✅ LightGBM Training Time: 85.29 sec


In [7]:
print("\nLightGBM Report (Train Set):")
print("Accuracy:", accuracy_score(y, y_pred_binary))
print(classification_report(y, y_pred_binary))
model.save_model("./models/lightgbm_model.txt")



LightGBM Report (Train Set):
Accuracy: 0.94526125
              precision    recall  f1-score   support

           0       0.95      0.95      0.95   2696949
           1       0.94      0.94      0.94   2103051

    accuracy                           0.95   4800000
   macro avg       0.94      0.94      0.94   4800000
weighted avg       0.95      0.95      0.95   4800000



In [8]:
nb = GaussianNB()
nb.fit(X, y)

y_pred = nb.predict(X)

print("Accuracy:", accuracy_score(y, y_pred))
print(classification_report(y, y_pred))

Accuracy: 0.7018510416666667
              precision    recall  f1-score   support

           0       0.66      0.98      0.79   2696949
           1       0.94      0.34      0.50   2103051

    accuracy                           0.70   4800000
   macro avg       0.80      0.66      0.64   4800000
weighted avg       0.78      0.70      0.66   4800000



In [9]:
print("\nTraining XGBoost (baseline params)...")
start = time.time()

xgb = XGBClassifier(
    n_estimators=200,       # number of trees (baseline)
    max_depth=3,            # depth of trees
    objective='binary:logistic',  # binary classification
    n_jobs=-1,
    random_state=42
)

xgb.fit(X, y)

end = time.time()
print(f"XGBoost Training Time: {end - start:.2f} sec")

# Predictions on train set
y_pred = xgb.predict(X)


Training XGBoost (baseline params)...
XGBoost Training Time: 17.82 sec


In [10]:
print("\nXGBoost Report (Train Set):")
print("Accuracy:", accuracy_score(y, y_pred))
print(classification_report(y, y_pred))
xgb.save_model("./models/xgboost_model_with_no_feature_filtration.json")


XGBoost Report (Train Set):
Accuracy: 0.9271175
              precision    recall  f1-score   support

           0       0.94      0.93      0.93   2696949
           1       0.91      0.92      0.92   2103051

    accuracy                           0.93   4800000
   macro avg       0.93      0.93      0.93   4800000
weighted avg       0.93      0.93      0.93   4800000



In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\nTraining MLP (baseline params)...")
start = time.time()

# Baseline MLP
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),  
    activation='relu',            
    solver='adam',                
    learning_rate_init=0.001,      
    max_iter=20,                 
    random_state=42,
    verbose=True                   
)

mlp.fit(X_scaled, y)

end = time.time()
print(f"✅ MLP Training Time: {end - start:.2f} sec")
y_pred = mlp.predict(X_scaled)


Training MLP (baseline params)...
Iteration 1, loss = 0.17726214
Iteration 2, loss = 0.15980241
Iteration 3, loss = 0.15307118
Iteration 4, loss = 0.15024718
Iteration 5, loss = 0.14829322
Iteration 6, loss = 0.14685627
Iteration 7, loss = 0.14561069
Iteration 8, loss = 0.14466075
Iteration 9, loss = 0.14385244
Iteration 10, loss = 0.14319878
Iteration 11, loss = 0.14261065
Iteration 12, loss = 0.14214779
Iteration 13, loss = 0.14169383
Iteration 14, loss = 0.14133346
Iteration 15, loss = 0.14103145
Iteration 16, loss = 0.14075896
Iteration 17, loss = 0.14053236
Iteration 18, loss = 0.14036345
Iteration 19, loss = 0.14007668
Iteration 20, loss = 0.13985862
✅ MLP Training Time: 1623.86 sec


C:\Users\soman\AppData\Roaming\Python\Python312\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


In [16]:
print("\nMLP Report (Train Set):")
print("Accuracy:", accuracy_score(y, y_pred))
print(classification_report(y, y_pred))


MLP Report (Train Set):
Accuracy: 0.9424202083333333
              precision    recall  f1-score   support

           0       0.95      0.95      0.95   2696949
           1       0.94      0.93      0.93   2103051

    accuracy                           0.94   4800000
   macro avg       0.94      0.94      0.94   4800000
weighted avg       0.94      0.94      0.94   4800000



In [18]:
joblib.dump(mlp, "./models/mlp_model.pkl")
joblib.dump(scaler, "./models/mlp_scaler.pkl")

['./models/mlp_scaler.pkl']